In [10]:
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd


In [11]:
ROOT = Path.cwd().resolve()
if ROOT.name == 'src':
    ROOT = ROOT.parent.parent
OUT = ROOT / 'b_simulate/output'
cfg = json.loads((ROOT / 'b_simulate/input/upstream/config.json').read_text())
SEED = cfg['seed']
streams = np.random.SeedSequence(SEED).spawn(1 + len(cfg['wind_blocks']))


In [12]:
def baseline_kernel(cfg):
    kernel = np.zeros((cfg['nx'], cfg['ny']))
    kernel[0, cfg['source_y']] = 1
    d = cfg['diffusivity']
    for x in range(1, cfg['nx']):
        kernel[x] = (1 - 2*d) * kernel[x - 1]
        kernel[x, 1:] += d * kernel[x - 1, :-1]
        kernel[x, :-1] += d * kernel[x - 1, 1:]
    return kernel

def balanced_wind(rng, n_trials, n_steps, block):
    signs = np.tile(np.repeat([-1, 1], n_steps // (2*block)), (n_trials, 1))
    signs = rng.permuted(signs, axis=1)
    signs = np.repeat(signs, block, axis=1)
    shifts = rng.integers(0, n_steps, size=n_trials)
    index = (np.arange(n_steps)[None, :] + shifts[:, None]) % n_steps
    return np.take_along_axis(signs, index, axis=1).astype(np.int8)


In [13]:
def step_ca(counts, sign, rng, cfg, bias):
    old = counts[:, :-1]
    p_up = cfg['diffusivity'] + bias * np.asarray(sign)[:, None, None] / 2
    p_down = cfg['diffusivity'] - bias * np.asarray(sign)[:, None, None] / 2
    up = rng.binomial(old, p_up)
    down = rng.binomial(old - up, p_down / (1 - p_up))
    east = old - up - down
    new = np.zeros_like(counts)
    new[:, 1:] = east
    new[:, 1:, 1:] += up[:, :, :-1]
    new[:, 1:, :-1] += down[:, :, 1:]
    new[:, 0, cfg['source_y']] = cfg['packets_per_step']
    lost = counts[:, -1].sum(axis=1) + up[:, :, -1].sum(axis=1) + down[:, :, 0].sum(axis=1)
    return new, lost

def initial_grid(rng, cfg, kernel):
    counts = np.zeros((cfg['n_trials'], cfg['nx'], cfg['ny']), dtype=np.int64)
    initial = np.array(['clean'] * (cfg['n_trials']//2) + ['established'] * (cfg['n_trials']//2))
    for x in range(cfg['nx']):
        counts[cfg['n_trials']//2:, x] = rng.multinomial(
            cfg['packets_per_step'], kernel[x], size=cfg['n_trials']//2)
    return counts, initial


In [14]:
def run_experiment(cfg, stream, block=None):
    wind_stream, packet_stream = stream.spawn(2)
    wind_rng = np.random.default_rng(wind_stream)
    packet_rng = np.random.default_rng(packet_stream)
    kernel = baseline_kernel(cfg)
    counts, initial = initial_grid(packet_rng, cfg, kernel)
    n = cfg['n_trials']
    measured_steps = cfg['baseline_steps'] if block is None else cfg['n_steps']
    total_steps = cfg['burn_in'] + measured_steps
    if block is None:
        wind = np.zeros((n, total_steps), dtype=np.int8)
        bias = 0.0
    else:
        measured = balanced_wind(wind_rng, n, cfg['n_steps'], block)
        startup = measured[:, -cfg['burn_in']:]
        wind = np.concatenate([startup, measured], axis=1)
        bias = cfg['wind_bias']
    trace = np.zeros((n, total_steps, len(cfg['receptor_offsets'])), dtype=np.float32)
    mean_grid = np.zeros_like(counts, dtype=np.float64)
    frames = []
    balance_error = np.zeros(n, dtype=np.int64)
    side_loss = np.zeros(n, dtype=np.int64)
    previous_mass = counts.sum(axis=(1, 2))
    for t in range(total_steps):
        counts, lost = step_ca(counts, wind[:, t], packet_rng, cfg, bias)
        mass = counts.sum(axis=(1, 2))
        error = mass - previous_mass - cfg['packets_per_step'] + lost
        balance_error = np.maximum(balance_error, np.abs(error))
        previous_mass = mass
        if t >= cfg['burn_in']:
            side_loss += lost - cfg['packets_per_step']
            mean_grid += counts
        for j, offset in enumerate(cfg['receptor_offsets']):
            center = cfg['source_y'] + offset
            h = cfg['receptor_halfwidth']
            trace[:, t, j] = counts[:, cfg['receptor_x'], center-h:center+h+1].mean(axis=1)
        if t % cfg['frame_interval'] == 0 and len(frames) < cfg['demo_frames']:
            frames.append(counts[0].astype(np.uint8))
    mean_grid /= measured_steps
    audit = pd.DataFrame({'trial': np.arange(n), 'initial': initial,
        'max_mass_balance_error': balance_error, 'lateral_loss_after_burnin': side_loss,
        'positive_wind_fraction': (wind[:, cfg['burn_in']:] == 1).mean(axis=1)})
    return {'mean_grid': mean_grid, 'trace': trace, 'wind': wind,
            'frames': np.stack(frames), 'initial': initial, 'audit': audit}


In [15]:
started = time.perf_counter()
baseline = run_experiment(cfg, streams[0])
np.savez_compressed(OUT / 'baseline.npz', **{k: v for k, v in baseline.items() if k != 'audit'})
audits = [baseline['audit'].assign(condition='baseline')]
runtimes = [{'condition': 'baseline', 'seconds': time.perf_counter() - started}]
for block, stream in zip(cfg['wind_blocks'], streams[1:]):
    started = time.perf_counter()
    result = run_experiment(cfg, stream, block)
    np.savez_compressed(OUT / f'wind_L{block}.npz', **{k: v for k, v in result.items() if k != 'audit'})
    audits.append(result['audit'].assign(condition=f'L={block}'))
    runtimes.append({'condition': f'L={block}', 'seconds': time.perf_counter() - started})
audit = pd.concat(audits, ignore_index=True)
audit.to_csv(OUT / 'mass_balance.csv', index=False)
(OUT / 'config.json').write_text(json.dumps(cfg, indent=2))
pd.DataFrame(runtimes)


,condition,seconds
0,baseline,12.200902
1,L=1,37.510948
2,L=8,37.565588
3,L=32,37.266928


In [16]:
assert audit['max_mass_balance_error'].max() == 0
assert audit['lateral_loss_after_burnin'].abs().max() == 0
audit.groupby('condition', sort=False).agg(
    trials=('trial', 'size'), max_mass_error=('max_mass_balance_error', 'max'),
    lateral_loss=('lateral_loss_after_burnin', 'sum'),
    mean_positive_wind_fraction=('positive_wind_fraction', 'mean'))


,trials,max_mass_error,lateral_loss,mean_positive_wind_fraction
condition,,,,
baseline,64,0,0,0.0
L=1,64,0,0,0.5
L=8,64,0,0,0.5
L=32,64,0,0,0.5


In [17]:

def step_weather(counts, drift, rng, cfg, source_x, source_y):
    d = cfg['diffusivity']
    p_east = d + drift[:, None, None]/2
    p_west = d - drift[:, None, None]/2
    east = rng.binomial(counts, p_east)
    remaining = counts - east
    west = rng.binomial(remaining, p_west/(1-p_east))
    remaining -= west
    north = rng.binomial(remaining, d/(1-p_east-p_west))
    remaining -= north
    south = rng.binomial(remaining, d/(1-p_east-p_west-d))
    new = remaining - south
    new[:, 1:] += east[:, :-1]
    new[:, :-1] += west[:, 1:]
    new[:, :, 1:] += north[:, :, :-1]
    new[:, :, :-1] += south[:, :, 1:]
    lost = east[:, -1].sum(axis=1) + west[:, 0].sum(axis=1)
    lost += north[:, :, -1].sum(axis=1) + south[:, :, 0].sum(axis=1)
    new[:, source_x, source_y] += cfg['packets_per_step']
    return new, lost


In [18]:
# two illustrative realizations, initially clean; no new exposure inference
weather_cfg = cfg['weather_demo']
weather_rng = np.random.default_rng(np.random.SeedSequence(SEED, spawn_key=(4,)))
weather_x = np.arange(weather_cfg['x_min'], weather_cfg['x_max']+1)
weather_y = np.arange(-weather_cfg['y_halfwidth'], weather_cfg['y_halfwidth']+1)
source_x = -weather_cfg['x_min']
source_y = weather_cfg['y_halfwidth']
drift = np.array([weather_cfg['gentle_drift'], 0.])
weather_counts = np.zeros((2, len(weather_x), len(weather_y)), dtype=np.int64)
weather_frames = []
weather_lost = np.zeros(2, dtype=np.int64)
weather_error = np.zeros(2, dtype=np.int64)
previous_mass = np.zeros(2, dtype=np.int64)
weather_steps = (cfg['demo_frames']-1)*cfg['frame_interval'] + 1
for t in range(weather_steps):
    weather_counts, lost = step_weather(weather_counts, drift, weather_rng, cfg, source_x, source_y)
    mass = weather_counts.sum(axis=(1, 2))
    weather_error = np.maximum(weather_error, np.abs(mass-previous_mass-cfg['packets_per_step']+lost))
    weather_lost += lost
    previous_mass = mass
    if t % cfg['frame_interval'] == 0:
        weather_frames.append(weather_counts.astype(np.int32))
np.savez_compressed(OUT / 'weather_demo.npz', frames=np.stack(weather_frames), x=weather_x, y=weather_y,
    drift=drift, names=np.array(['gentle', 'calm']), steps=np.arange(0, weather_steps, cfg['frame_interval'])+1)
weather_audit = pd.DataFrame({'condition': ['gentle', 'calm'], 'steps': weather_steps,
    'mean_eastward_drift': drift, 'emitted': cfg['packets_per_step']*weather_steps,
    'remaining': previous_mass, 'exited': weather_lost, 'max_mass_balance_error': weather_error})
assert weather_error.max() == 0
weather_audit.to_csv(OUT / 'weather_demo_audit.csv', index=False)
(OUT / 'config.json').write_text(json.dumps(cfg, indent=2))
weather_audit


,condition,steps,mean_eastward_drift,emitted,remaining,exited,max_mass_balance_error
0,gentle,1433,0.15,114640,76821,37819,0
1,calm,1433,0.00,114640,114579,61,0
